# capybucks — getting started

This notebook walks the public API: `download()`, `Ticker`, provenance, errors, B3, crypto, and indicators.

**Needs the network.** Equities default to Yahoo (unofficial, usually delayed). Crypto uses CoinGecko. Optional Twelve Data cells skip if you have no key.

Install: `pip install capybucks`, or from a clone `uv sync --all-groups` and pick that interpreter as the kernel.

Full prose: [docs](https://witmondcal-maker.github.io/capybucks/).

In [ ]:
import capybucks as cb

df = cb.download("AAPL", period="1mo")
print(df.attrs["provider"], df.attrs["delayed"], df.attrs.get("currency"))
df.tail()

Columns are OHLCV. Provenance lives on `df.attrs`, not as extra columns. `delayed=True` means this must not be treated as a live print.

In [ ]:
list(df.columns), type(df.index), str(df.index.tz)

## Batch and partial failure

A dead ticker does not block the others. Failures show up in `attrs["errors"]`. If every name fails, you get `MultiDownloadError` instead of an empty frame.

In [ ]:
batch = cb.download(["AAPL", "NOTAREALTICKER", "MSFT"], period="5d", group_by="column")
print(batch.attrs["errors"])
batch["Close"].tail()

With several tickers, columns are a MultiIndex (`group_by="ticker"` by default). `group_by="column"` puts `Close` on top so `frame["Close"]` is a wide DataFrame of names.

## Ticker: financials, options, actions, news

Same Yahoo path as `download()` unless you pass `provider=`. Properties hit the network on each access — bind the result if you need it twice.

In [ ]:
aapl = cb.Ticker("AAPL")
hist = aapl.history(period="5d")
inc = aapl.financials
print("history rows", len(hist), "income items", list(inc.index)[:8])
inc.iloc[:5, :4]

In [ ]:
print(aapl.options[:5])
chain = aapl.option_chain()
chain.calls.head()

In [ ]:
display(aapl.dividends.tail())
display(aapl.news.head())

## Errors are typed

Missing names raise `SymbolNotFound`. Brazilian cash tickers without `.SA` get a hint rather than a silent rewrite.

In [ ]:
from capybucks.exceptions import SymbolNotFound

try:
    cb.download("PETR4", period="5d")
except SymbolNotFound as err:
    print(err)
    print("hint:", err.hint)

petr = cb.download("PETR4.SA", period="5d")
print(petr.attrs.get("currency"), petr.index.tz)
petr.tail(3)

## Crypto

`BTC-USD` (and similar) resolve to CoinGecko, not Yahoo.

In [ ]:
btc = cb.download("BTC-USD", period="5d")
print(btc.attrs["provider"])
btc.tail()

## Indicators

`capybucks.ta` is pandas-only. It does not fetch; it transforms the OHLCV you already have.

In [ ]:
from capybucks.ta import enrich, rsi

six = cb.download("AAPL", period="6mo")
print(rsi(six["Close"]).tail())
enrich(six, window=20).tail()[["Close", "SMA", "RSI", "VWAP"]]

## Paid provider (optional)

Twelve Data is pin-only. Without a key this cell skips. Set `CAPYBUCKS_TWELVEDATA_KEY` or `~/.capybucks.toml` — see [configuration](../docs/configuration.md). A `.env` file is **not** auto-loaded; in a notebook, export the env in the kernel or use `os.environ[...] = ...` in a private cell you do not commit.

In [ ]:
import os

from capybucks.exceptions import MissingAPIKey

if os.environ.get("CAPYBUCKS_TWELVEDATA_KEY"):
    td = cb.download("AAPL", period="5d", provider="twelvedata", cache=False)
    print(td.attrs["provider"], len(td))
    display(td.tail())
else:
    try:
        cb.download("AAPL", period="5d", provider="twelvedata")
    except MissingAPIKey as err:
        print("expected without a key:", err)

There is no silent fallback across vendors. If you need another source, pass `provider=` yourself.

Next: [docs/download.md](../docs/download.md), [docs/ticker.md](../docs/ticker.md), [docs/errors.md](../docs/errors.md).